### Decision tree splitting — formulas, derived

**Gini impurity of one group**
Gini = 1 - sum(p_c^2) over classes c
Binary case: Gini = 1 - (p^2 + (1-p)^2). Lower = purer.
- 50/50 mix: 1-(0.5^2+0.5^2) = 0.5 (max impurity for 2 classes)
- 90/10 mix: 1-(0.9^2+0.1^2) = 0.18
- pure (100/0): 1-(1^2+0^2) = 0
General k-class max impurity = 1 - 1/k (grows with more classes).

**Scoring a candidate split: weighted Gini**
A split produces two groups. Weight each side's Gini by its share of the data, so a split that
carves off one tiny pure sliver but leaves the rest untouched isn't rewarded:
weighted_gini = (n_left/n_total)*Gini(left) + (n_right/n_total)*Gini(right)
Best split = whichever (feature, threshold) candidate gives the LOWEST weighted_gini.

**Candidate thresholds**
For a numeric feature, candidates are midpoints between consecutive sorted unique values --
avoids a threshold landing exactly on an observed value, which would be ambiguous (which side
does an exact match fall on?).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
n = 100

# XOR: no straight line separates this; only x AND y together carry signal
# class 0: top-right and bottom-left quadrants
c0a = rng.normal(loc=[2, 2], scale=0.5, size=(n, 2))
c0b = rng.normal(loc=[-2, -2], scale=0.5, size=(n, 2))
# class 1: top-left and bottom-right quadrants
c1a = rng.normal(loc=[-2, 2], scale=0.5, size=(n, 2))
c1b = rng.normal(loc=[2, -2], scale=0.5, size=(n, 2))

X_tree = np.vstack([c0a, c0b, c1a, c1b])
y_tree = np.hstack([np.zeros(2*n), np.ones(2*n)])

plt.scatter(X_tree[:, 0], X_tree[:, 1], c=y_tree, cmap="coolwarm", alpha=0.6)
plt.title("XOR-pattern toy data — no straight line can separate this")
plt.show()

In [ ]:
# Gini = 1 - sum(p_c^2); binary, so mean(y) = p(class 1)
def gini(y):
    if len(y) == 0:
        return 0
    p = np.mean(y)
    return 1 - (p**2 + (1 - p)**2)

print("gini of pure group [1,1,1,1]:", gini(np.array([1,1,1,1])))   # expect 0
print("gini of 50/50 group [1,1,0,0]:", gini(np.array([1,1,0,0])))  # expect 0.5
print("gini of full y_tree (50/50 by construction):", gini(y_tree)) # expect 0.5


In [ ]:
# tries every (feature, threshold) candidate, scores with weighted gini, keeps the best
def best_split(X, y):
    best_gini = float("inf")
    best_feature, best_threshold = None, None
    n_total = len(y)

    for feature_idx in range(X.shape[1]):
        values = np.sort(np.unique(X[:, feature_idx]))  # np.sort redundant, unique already sorts
        thresholds = (values[:-1] + values[1:]) / 2     # midpoints, avoids landing on a data point

        for t in thresholds:
            left_mask = X[:, feature_idx] <= t
            y_left, y_right = y[left_mask], y[~left_mask]

            if len(y_left) == 0 or len(y_right) == 0:
                continue

            weighted_gini = (len(y_left)/n_total) * gini(y_left) + (len(y_right)/n_total) * gini(y_right)

            if weighted_gini < best_gini:
                best_gini = weighted_gini
                best_feature, best_threshold = feature_idx, t

    return best_feature, best_threshold, best_gini

feature, threshold, g = best_split(X_tree, y_tree)
print(f"best split: feature={feature}, threshold={threshold:.3f}, weighted gini={g:.3f}")
# ~0.496, barely better than 0.5 -- true XOR has no informative single split


In [ ]:
left_mask = X_tree[:, feature] <= threshold
X_left, y_left = X_tree[left_mask], y_tree[left_mask]
X_right, y_right = X_tree[~left_mask], y_tree[~left_mask]

print("left: n =", len(y_left), ", class balance =", y_left.mean(), ", gini =", gini(y_left))
print("right: n =", len(y_right), ", class balance =", y_right.mean(), ", gini =", gini(y_right))

f_left, t_left, g_left = best_split(X_left, y_left)
f_right, t_right, g_right = best_split(X_right, y_right)
print(f"left's best next split: feature={f_left}, threshold={t_left:.3f}, weighted gini={g_left:.3f}")
print(f"right's best next split: feature={f_right}, threshold={t_right:.3f}, weighted gini={g_right:.3f}")

# right = 12 pts, barely skewed -- noise, not signal. left still ~unsolved (gini 0.5)
# greedy trees have no lookahead -- can't solve pure XOR in 2 splits, ties broken by noise
# real fraud interactions aren't this adversarial: amt alone already carried real signal
